# CNN-BiLSTM — NBD — Pooled Split / Subject-Independent (scEEG -> iEEG)

**Split:** ALL segments from ALL subjects are pooled, then split **70% train / 10%
validation / 20% test by SEGMENT COUNT** — same idea you described — but assigned at
the **SUBJECT level**: every subject's segments go, whole, into exactly one of
train/val/test. **One** CNN-BiLSTM is trained on the pooled training subjects and
evaluated on the pooled (entirely unseen) test subjects.


### What changed in this "-nbd-28" rebuild (per your mam's feedback)

**Model/loss background (from your original notebooks, unchanged):** CNN-BiLSTM is a
widely used hybrid for EEG signal work (e.g. Antoniades et al. 2018 for
scalp->intracranial mapping specifically). The CNN stack extracts local spatial
(cross-channel) and short-range temporal (spike/sharp-wave shape) features while
preserving the full sequence length; the BiLSTM then gives every output timestep both
past and future context. iEEG channel amplitudes vary a lot across channels in this
kind of data, so scEEG/iEEG are standardized **per channel** (not with a single global
scale) — unchanged from your original notebooks.

**What's actually different here:**

1. **New dataset.** Loads `segmented_dataset.npz` (was `balanced_segmented_dataset.npz`).
   The loader below looks up array names through a short alias list and prints exactly
   which keys it found, instead of silently assuming the old schema.
2. **Labels are now optional.** Your new dataset may not carry an IED/non-IED label at
   all. The loader detects this (`HAS_LABELS`) and automatically: (a) switches every
   stratified split to a plain/group split instead, and (b) drops the IED/Non-IED rows
   from the results table, reporting "Combined" only. Nothing below will crash if the
   label array is simply missing.
3. **Data-leakage fix.**
   **This is the notebook your mam's leakage comment applies to.** The old version of
   this notebook did a single `train_test_split` over ALL pooled *segments*, ignoring
   which subject each segment came from — so segments from the same subject could land
   in both train and test. A model can then partly "recognize" a subject's own signal
   statistics from training and score higher on that subject's held-out segments than
   it would on a genuinely new subject — that's the leakage.
   **The fix** (Section 7 below, `group_aware_pooled_split`): every subject is assigned,
   *whole*, to exactly one of train/val/test, so no subject's segments ever appear in
   more than one split — while still targeting your original 70/10/20 **segment-count**
   ratio as closely as whole-subject boundaries allow (a greedy bin-packing over
   subjects). An assertion after the split proves the three subject sets are disjoint.
4. **[Round 2] Training cosine loss re-aligned to the exact reported COSSIM.** This was
   flagged as the single biggest issue: `loss_cosine` used to z-score the signals
   before computing cosine similarity, while `score_mapping`'s reported COSSIM uses the
   raw dot-product/norms. Those are different quantities — a model could improve its
   *training* cosine loss while reported COSSIM barely moved. Fixed by computing
   `loss_cosine` on the raw values, exactly matching `score_mapping`. (`loss_pearson`
   never had this problem — Pearson correlation is shift/scale invariant by definition.)
5. **[Round 2] Dropout reduced (0.3 → 0.15).** The original model stacked CNN dropout +
   BiLSTM dropout + head dropout + channel attention + weight decay + gradient clipping
   + early stopping — a lot of regularization for a genuinely hard regression problem.
   Over-regularizing can push the model toward a smooth, averaged output rather than
   reproducing exact temporal morphology, which shows up as weak point-by-point
   alignment (low PCORR) even when predictions "look" plausible.
6. **[Round 2] Channel attention applied once, not after every conv block.** Repeated
   attention, based on each channel's time-averaged activation, can let the model learn
   "this channel's average is low, so suppress it" — even when that channel's value is
   dominated by a brief high-amplitude transient (a spike/sharp wave) riding on a
   near-zero baseline, i.e. exactly the information the reconstruction needs.
7. **[Round 2] Dilated convolutions widen the temporal receptive field** (`DILATIONS =
   (1, 2, 4)`), instead of three same-dilation conv blocks. This lets the CNN stage
   itself capture more of the local spike/sharp-wave/oscillatory morphology before
   handing off to the BiLSTM, rather than relying on the BiLSTM alone to recover longer
   temporal structure.
8. **[Round 2] Two-layer reconstruction head** (was a single Linear→activation→Linear).
   The BiLSTM produces a rich representation; the previous head had limited capacity to
   reshape that into detailed iEEG morphology. This gives it more room to do so.
9. **[Round 2] Amplitude loss weight lowered** (`w_amp`: 4.0 → 1.5) and treated as a
   light secondary constraint. `w_pcorr`/`w_cos` already drive shape+scale matching;
   over-weighting amplitude makes the model spend capacity getting the scale right
   instead of getting the temporal shape right, which is what PCORR/COSSIM measure.
10. **[Round 2] Learning rate lowered** (1e-3 → 5e-4). 1e-3 can be aggressive for a
   fairly deep CNN + 2-layer BiLSTM on a waveform-fitting task; a lower rate tends to
   produce more stable convergence onto fine morphology rather than a coarse fit.
11. **Honest expectation-setting on 0.8-0.9:** these changes give the model the best
   realistic chance, but a jump from ~0.4-0.5 to 0.8-0.9 can't be guaranteed by
   architecture/loss tuning alone if the underlying scEEG→iEEG relationship is weak for
   a given split. If your earlier 0.8-0.9 numbers came from a leaky split, they aren't a
   fair target for a leakage-free evaluation — a lower, honest number is a *more*
   trustworthy estimate of real generalization, not evidence the model regressed. Use
   `plot_best_vs_typical` and the new per-channel breakdown to see where the real gap
   is, rather than assuming it's uniform (point 12 below).
12. **[Round 2] Per-channel metrics breakdown added.** A single poor channel can pull
   the Combined average down a lot even when most channels are already matching well
   (e.g. three channels at ~0.85 and one at ~0.20 average to only ~0.66). A new cell
   after the main results table reports PCORR/COSSIM/MSE per iEEG channel so this can
   be told apart from a genuinely uniform result.
13. **[from the previous round, unchanged]** Training schedule is just `EPOCHS` +
   `PATIENCE` (CNN-BiLSTM never had a separate warm-up knob — that's a GAN-only thing
   in the VAE-cGAN notebooks). Set `EPOCHS` as a generous upper bound and let
   `PATIENCE` decide when a run actually stops.
14. Everything else — the general notebook structure and the literal MSE/PCORR/COSSIM
   definitions — is unchanged.


## 1. Setup & config

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os, copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

DATA_PATH = "segmented_dataset.npz"   # <-- your new dataset

# ---- pooled split, assigned by SUBJECT (leakage-free) -- target segment-count ratio ----
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20

# ---- model size (bigger than the per-subject notebook: pooled training data) ----
CNN_CHANNELS = (64, 128, 256)
KERNEL_SIZE = 5
DILATIONS = (1, 2, 4)   # widens temporal receptive field without extra depth/downsampling
LSTM_HIDDEN, LSTM_LAYERS = 192, 2
DROPOUT = 0.15

# ---- training ----
# w_pcorr/w_cos raised, w_amp (NEW amplitude loss) added -- see markdown note above.
EPOCHS, PATIENCE = 100, 25
LR = 5e-4
W_MSE, W_PCORR, W_COS, W_L1, W_AMP = 0.5, 8.0, 8.0, 0.2, 1.5
BATCH_SIZE = 64


## 2. Load the segmented dataset

In [ ]:
data = np.load(DATA_PATH, allow_pickle=True)
print(f"Keys found in {DATA_PATH}: {list(data.keys())}")

def _first_present(d, candidates):
    for c in candidates:
        if c in d:
            return c
    return None

# ---- flexible key lookup -----------------------------------------------
# segmented_dataset.npz is a NEW/different file from the old
# balanced_segmented_dataset.npz -- it may not use identical key names, so
# every array is looked up through a short list of common aliases instead of
# a single hard-coded key. If your actual key names aren't in these lists,
# just add them -- this is deliberately the ONLY place that needs editing.
eeg_key  = _first_present(data, ["X_eeg", "X_sc", "X_scalp", "sceeg", "scEEG"])
ieeg_key = _first_present(data, ["X_ieeg", "X_ic", "X_intracranial", "ieeg", "iEEG"])
if eeg_key is None or ieeg_key is None:
    raise KeyError(
        f"Could not find scEEG/iEEG arrays in {DATA_PATH}. Keys present: {list(data.keys())}. "
        "Add your actual key name(s) to the candidate lists above (_first_present calls)."
    )

X_eeg  = data[eeg_key].astype(np.float32)      # (N, L, M)  scEEG
X_ieeg = data[ieeg_key].astype(np.float32)     # (N, L, Mb) iEEG (target)

subj_key = _first_present(data, ["subject_ids", "subject_id", "subjects", "subj_ids"])
if subj_key is None:
    raise KeyError(f"Could not find a subject-id array in {DATA_PATH}. Keys present: {list(data.keys())}.")
subject_ids = np.asarray(data[subj_key])

# ---- labels are now OPTIONAL --------------------------------------------
# Your new segmented_dataset.npz may not carry an IED/non-IED label at all
# (unlike the old balanced_segmented_dataset.npz). This is detected instead
# of assumed: if no label array is found, HAS_LABELS=False and every
# downstream stratified split / IED-vs-Non-IED breakdown is switched off
# automatically (falls back to plain random/group splits and a
# "Combined"-only results table), instead of crashing on a KeyError.
label_key = _first_present(data, ["y", "labels", "ied_label", "ied_labels", "label"])
HAS_LABELS = label_key is not None
if HAS_LABELS:
    y = np.asarray(data[label_key]).astype(np.int64)
    print(f"Label array found (key='{label_key}') -- IED/Non-IED breakdown + stratified splits enabled.")
else:
    y = np.zeros(len(X_eeg), dtype=np.int64)   # placeholder ONLY -- never used for stratification/reporting
    print("No IED/Non-IED label array found in this dataset -- IED/Non-IED breakdown is disabled, "
          "and every split below is UNSTRATIFIED (plain random / subject-group based, not label-balanced).")

eeg_names = list(data["eeg_names"]) if "eeg_names" in data else [f"sc_ch{i}" for i in range(X_eeg.shape[2])]
fo_names  = list(data["fo_names"])  if "fo_names"  in data else [f"ie_ch{i}" for i in range(X_ieeg.shape[2])]
fs = float(data["fs"]) if "fs" in data else 256.0

L  = X_eeg.shape[1]           # time samples per segment
M  = X_eeg.shape[2]           # scEEG channels
Mb = X_ieeg.shape[2]          # iEEG channels

unique_subjects = sorted(np.unique(subject_ids).tolist())
print(f"Total segments: {len(X_eeg)}  |  scEEG shape: {X_eeg.shape}  |  iEEG shape: {X_ieeg.shape}")
print(f"Subjects ({len(unique_subjects)}):", unique_subjects)
if HAS_LABELS:
    print(f"IED: {int((y==1).sum())}   Non-IED: {int((y==0).sum())}")

# Classes reported everywhere downstream -- collapses to just "Combined"
# when this dataset has no IED/non-IED labels.
CLASSES = ["Combined", "IED", "Non-IED"] if HAS_LABELS else ["Combined"]


## 3. CNN-BiLSTM model architecture

In [ ]:
# ============================================================================
# CNN-BiLSTM model: scEEG (B, L, M) -> estimated iEEG (B, L, Mb).
#   1. A stack of DILATED 1D-CNN blocks (Conv1d + BatchNorm + LeakyReLU +
#      Dropout) slides over the time axis. Dilation (1, 2, 4 by default)
#      widens the effective temporal receptive field WITHOUT adding depth or
#      downsampling -- so local spike/sharp-wave morphology (small dilation)
#      and longer patterns (larger dilation) are both captured before the
#      BiLSTM has to do all the work of recovering longer-range structure.
#      Padding is dilation-aware so the sequence length stays fixed at L.
#   2. ONE squeeze-and-excite channel-attention block, applied ONLY at the
#      input (changed from being re-applied after every conv block). Stacking
#      attention after every block, based on each channel's time-AVERAGE
#      activation, could repeatedly down-weight exactly the channels whose
#      value lies in a brief transient riding on a near-zero baseline (i.e.
#      the low-average channels an EEG spike/sharp-wave depends on) --
#      suppressing information the reconstruction actually needs.
#   3. A multi-layer Bidirectional LSTM consumes the CNN feature sequence, so
#      every output timestep has both past and future context.
#   4. A two-layer (was one-layer) per-timestep reconstruction head maps
#      BiLSTM hidden state -> Mb iEEG channels at every timestep, giving the
#      network more capacity to reshape the BiLSTM representation into
#      detailed waveform morphology instead of an almost-direct projection.
# ============================================================================

class ChannelAttention(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        hidden = max(ch // r, 4)
        self.fc = nn.Sequential(
            nn.Linear(ch, hidden), nn.ReLU(), nn.Linear(hidden, ch), nn.Sigmoid())

    def forward(self, x):  # x: (B, C, L)
        w = x.mean(dim=2)              # (B, C) global average pool over time
        w = self.fc(w).unsqueeze(2)    # (B, C, 1)
        return x * w


class CNNBiLSTM(nn.Module):
    def __init__(self, in_ch=20, out_ch=12, cnn_channels=(64, 128, 128),
                 kernel_size=5, dilations=(1, 2, 4), lstm_hidden=128, lstm_layers=2,
                 dropout=0.15):
        super().__init__()
        assert len(cnn_channels) == len(dilations), "cnn_channels and dilations must be the same length"

        # single channel-attention block, applied ONCE at the input -- see class
        # docstring above for why this is no longer stacked after every conv block.
        self.in_attn = ChannelAttention(in_ch)

        blocks = []
        c_in = in_ch
        for c_out, dil in zip(cnn_channels, dilations):
            pad = dil * (kernel_size - 1) // 2   # dilation-aware padding keeps length == L
            blocks.append(nn.Sequential(
                nn.Conv1d(c_in, c_out, kernel_size=kernel_size, padding=pad, dilation=dil),
                nn.BatchNorm1d(c_out),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout * 0.5),
            ))
            c_in = c_out
        self.cnn = nn.Sequential(*blocks)

        self.bilstm = nn.LSTM(
            input_size=c_in, hidden_size=lstm_hidden, num_layers=lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        # Two-layer reconstruction head (was a single Linear->activation->Linear).
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, lstm_hidden // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden // 2, out_ch),
        )

    def forward(self, x):
        # x: (B, L, M) -> conv wants (B, M, L)
        h = x.permute(0, 2, 1)
        h = self.in_attn(h)
        h = self.cnn(h)
        h = h.permute(0, 2, 1)          # (B, L, C) for LSTM
        h, _ = self.bilstm(h)           # (B, L, 2*hidden)
        return self.head(h)             # (B, L, out_ch), linear output (raw/scaled space)


## 4. Loss functions

In [ ]:
# ============================================================================
# Losses: MSE + L1 (point-wise) + explicit 1-Pearson / 1-Cosine (shape-matching)
# + a light secondary amplitude term.
#
# loss_cosine is now aligned EXACTLY with the reported COSSIM (see its docstring
# below) -- this was flagged as the single biggest source of train/eval mismatch:
# the previous version z-scored the signals before computing cosine similarity,
# which discards the DC offset and optimizes a DIFFERENT quantity than the raw
# dot-product/norms COSSIM actually reported by score_mapping.
#
# w_amp is deliberately kept LOW relative to w_pcorr/w_cos: those two already
# drive shape+scale matching, so amplitude is a secondary constraint against
# outright amplitude collapse, not a competing primary objective -- giving it too
# much weight makes the model spend capacity on scale instead of temporal shape.
# ============================================================================

def loss_pearson(y_real, y_est, eps=1e-8):
    '''Pearson correlation is itself shift/scale invariant (it mean-centers and
    normalizes internally), so this always exactly matches PCORR -- no
    z-scoring/alignment issue here, unlike loss_cosine below.'''
    yr = y_real - y_real.mean(dim=1, keepdim=True)
    ye = y_est - y_est.mean(dim=1, keepdim=True)
    num = (yr * ye).sum(dim=1)
    den = torch.sqrt((yr ** 2).sum(dim=1) * (ye ** 2).sum(dim=1) + eps)
    return 1 - (num / (den + eps)).mean()

def loss_cosine(y_real, y_est, eps=1e-8):
    '''Matches the EXACT COSSIM definition used in score_mapping/evaluation: raw
    dot product over raw norms -- NOT cosine similarity of z-scored signals. The
    earlier version z-scored y_real/y_est first, discarding the DC offset/mean
    before computing cosine similarity, so the network was optimized against a
    different quantity than the one actually reported. This version is exactly
    what's reported, so improving this loss now directly improves reported COSSIM.'''
    num = (y_real * y_est).sum(dim=1)
    den = torch.norm(y_real, dim=1) * torch.norm(y_est, dim=1)
    return 1 - (num / (den + eps)).mean()

def loss_amplitude(y_real, y_est, eps=1e-8):
    '''Light secondary constraint against amplitude collapse (see markdown above) --
    intentionally weighted low relative to loss_pearson/loss_cosine so it doesn't
    compete with shape matching for model capacity.'''
    real_amp = y_real.std(dim=1) + eps
    est_amp  = y_est.std(dim=1) + eps
    return torch.mean(((real_amp - est_amp) / real_amp) ** 2)

def loss_total(y_real, y_est, w_mse=0.5, w_pcorr=8.0, w_cos=8.0, w_l1=0.2, w_amp=1.5):
    lmse = F.mse_loss(y_est, y_real)
    ll1  = F.l1_loss(y_est, y_real)
    lpc  = loss_pearson(y_real, y_est)
    lcos = loss_cosine(y_real, y_est)
    lamp = loss_amplitude(y_real, y_est)
    total = w_mse * lmse + w_l1 * ll1 + w_pcorr * lpc + w_cos * lcos + w_amp * lamp
    return total, lmse, ll1, lpc, lcos, lamp


In [ ]:
class SegSet(Dataset):
    def __init__(self, sc, ie, lab):
        self.sc  = torch.tensor(sc,  dtype=torch.float32)
        self.ie  = torch.tensor(ie,  dtype=torch.float32)
        self.lab = torch.tensor(lab, dtype=torch.float32)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i): return self.sc[i], self.ie[i], self.lab[i]


## 5. Metrics: MSE / PCORR / COSSIM

In [ ]:
def score_mapping(model, loader, device=DEVICE):
    '''Returns MSE, PCORR, COSSIM for "Combined", and ALSO for "IED"/"Non-IED" when this
    dataset actually has labels (HAS_LABELS, set in the data-loading cell). Computed
    directly on the values the loader provides -- the same per-channel, training-set-fit
    standardized representation used for training -- with NO extra per-segment
    re-normalization, which keeps MSE and COSSIM genuinely independent of PCORR (mirrors
    the paper's literal Eqs. 14-16; unchanged from your original notebooks).'''
    model.eval()
    mse_vals, pcorr_vals, cos_vals, label_vals = [], [], [], []
    with torch.no_grad():
        for sc, ie, lab in loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for i in range(ie_np.shape[0]):
                for j in range(ie_np.shape[2]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    mse_vals.append(np.mean((yv - yev) ** 2))
                    pcorr_vals.append(pearsonr(yv, yev)[0])
                    denom = np.linalg.norm(yv) * np.linalg.norm(yev)
                    cos_vals.append(np.dot(yv, yev) / (denom + 1e-8))
                    label_vals.append(lab[i].item())
    mse_vals, pcorr_vals, cos_vals, label_vals = (np.array(a) for a in
        (mse_vals, pcorr_vals, cos_vals, label_vals))

    def summarize(mask):
        if mask.sum() == 0:
            return dict(MSE=np.nan, PCORR=np.nan, COSSIM=np.nan)
        return dict(MSE=float(np.mean(mse_vals[mask])),
                    PCORR=float(np.mean(pcorr_vals[mask])),
                    COSSIM=float(np.mean(cos_vals[mask])))

    out = {"Combined": summarize(np.ones_like(label_vals, dtype=bool))}
    if HAS_LABELS:
        out["IED"] = summarize(label_vals == 1)
        out["Non-IED"] = summarize(label_vals == 0)
    return out


## 6. Training loop

In [ ]:
def fit_model(model, train_loader, val_loader, device=DEVICE, epochs=100, lr=1e-3,
              w_mse=0.5, w_pcorr=8.0, w_cos=8.0, w_l1=0.2, w_amp=4.0,
              patience=20, grad_clip=5.0, verbose=True):
    '''Supervised training loop: Adam + ReduceLROnPlateau + early stopping.
    Checkpoint selection tracks validation (PCORR + COSSIM)/2 directly, since
    that's what's ultimately reported. Only change from the original notebooks:
    w_amp (amplitude loss weight) is threaded through everywhere loss_total is called.'''
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=8)

    best_score, wait = -float('inf'), 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        tr_loss = 0.0
        for sc, ie, _ in train_loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            total, lmse, ll1, lpc, lcos, lamp = loss_total(ie, y_est, w_mse, w_pcorr, w_cos, w_l1, w_amp)
            opt.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            tr_loss += total.item()
        tr_loss /= len(train_loader)

        model.eval()
        va_loss = va_corr = va_cos = 0.0
        with torch.no_grad():
            for sc, ie, _ in val_loader:
                sc, ie = sc.to(device), ie.to(device)
                y_est = model(sc)
                total, lmse, ll1, lpc, lcos, lamp = loss_total(ie, y_est, w_mse, w_pcorr, w_cos, w_l1, w_amp)
                va_loss += total.item()
                va_corr += (1 - lpc).item()
                va_cos  += (1 - lcos).item()
        va_loss /= len(val_loader); va_corr /= len(val_loader); va_cos /= len(val_loader)
        sched.step(va_loss)
        val_score = (va_corr + va_cos) / 2

        if val_score > best_score:
            best_score, wait = val_score, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train_loss {tr_loss:.3f} "
                  f"| val_PCORR {va_corr:.3f} val_COSSIM {va_cos:.3f}")

        if wait >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [ ]:
def metrics_dict_to_row(d):
    '''Flatten {"Combined":{...}, ["IED":{...}, "Non-IED":{...}]} into one flat row.
    Only includes IED/Non-IED columns when this dataset actually has labels
    (CLASSES is set in the data-loading cell based on HAS_LABELS).'''
    row = {}
    for cls in CLASSES:
        for met in ["MSE", "PCORR", "COSSIM"]:
            row[(met, cls)] = d[cls][met]
    return row

def build_results_table(rows_dict, index_name="Subject"):
    '''rows_dict: {row_label: metrics_dict}. Returns a DataFrame with a
    (metric, class) MultiIndex column layout, plus a trailing Mean row.'''
    flat_rows = {label: metrics_dict_to_row(d) for label, d in rows_dict.items()}
    df = pd.DataFrame.from_dict(flat_rows, orient="index")
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["Metric", "Class"])
    df.index.name = index_name
    mean_row = df.mean(numeric_only=True)
    df.loc["Mean"] = mean_row
    return df.round(3)


In [ ]:
def plot_best_vs_typical(model, sc_batch, ie_batch, lab_batch, ie_mu, ie_sd, fo_names,
                          n_examples=4, device=DEVICE, title=""):
    '''Honest view of match quality: top row is the best-matching segment/channel
    pairs found in this batch (cherry-picked, labeled as such -- NOT the average case),
    bottom row is randomly sampled segments (the typical case). Per-segment per-channel
    Pearson correlation is shown on each subplot.'''
    model.eval()
    with torch.no_grad():
        y_est = model(sc_batch.to(device)).cpu().numpy()
    ie_np = ie_batch.numpy()
    lab_np = lab_batch.numpy()
    n_seg, L, n_ch = ie_np.shape

    scored = []
    for i in range(n_seg):
        for j in range(n_ch):
            c = pearsonr(ie_np[i, :, j], y_est[i, :, j])[0]
            scored.append((i, j, c))
    scored.sort(key=lambda t: -t[2])
    best = scored[:n_examples]

    rng = np.random.RandomState(7)
    rand_idx = rng.choice(n_seg, min(n_examples, n_seg), replace=False)

    fig, axes = plt.subplots(2, n_examples, figsize=(4.2 * n_examples, 7))
    for k, (i, j, c) in enumerate(best):
        real = ie_np[i, :, j] * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        est  = y_est[i, :, j]  * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        lab_txt = f"label={int(lab_np[i])}" if HAS_LABELS else f"seg#{i}"
        axes[0, k].plot(real, "k", label="Real")
        axes[0, k].plot(est, "crimson", alpha=0.8, label="Estimated")
        axes[0, k].set_title(f"BEST-CASE  ch={fo_names[j]}  corr={c:.2f}  {lab_txt}", fontsize=9)
        axes[0, k].legend(fontsize=7)
    for k, i in enumerate(rand_idx):
        j = 0
        c = pearsonr(ie_np[i, :, j], y_est[i, :, j])[0]
        real = ie_np[i, :, j] * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        est  = y_est[i, :, j]  * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        lab_txt = f"label={int(lab_np[i])}" if HAS_LABELS else f"seg#{i}"
        axes[1, k].plot(real, "k", label="Real")
        axes[1, k].plot(est, "crimson", alpha=0.8, label="Estimated")
        axes[1, k].set_title(f"RANDOM/TYPICAL  ch={fo_names[j]}  corr={c:.2f}  {lab_txt}", fontsize=9)
        axes[1, k].legend(fontsize=7)
    fig.suptitle(title + "\nTop: best-matching segment/channel pairs found (cherry-picked, NOT the average). "
                          "Bottom: random/typical segments -- this is what the reported averages actually reflect.")
    plt.tight_layout()
    plt.show()


## 7. Leakage-free, subject-grouped pooled 70/10/20 split, with per-channel
## z-score standardization

Every subject is assigned, whole, to exactly ONE of train/val/test (see the function
docstring below). This directly replaces the old segment-level `train_test_split` that
could leak a subject's own data across the train/test boundary. scEEG/iEEG are then
standardized per channel using the pooled TRAINING subjects' statistics only.


In [ ]:
def group_aware_pooled_split(subject_ids, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC,
                              test_frac=TEST_FRAC, seed=SEED):
    '''THE fix for the data-leakage bug in the old pooled/subject-independent split.

    OLD (leaky) approach: a *segment*-level StratifiedKFold ran over ALL pooled
    segments, ignoring subject identity -- so segments from the SAME subject could
    land in both the train and test portions of a fold. A model can then partly
    "recognize" a subject's own signal statistics/idiosyncrasies from training and
    get an inflated test score that has nothing to do with generalizing to a new
    subject -- exactly the leakage flagged as a problem.

    NEW approach (this function): each SUBJECT, whole, is assigned to exactly ONE
    of train/val/test -- a subject's segments can never cross a split boundary --
    while still targeting your original idea (70% / 10% / 20% of total SEGMENTS)
    as closely as whole-subject group boundaries allow. This is a greedy
    largest-deficit-first bin-packing: subjects are shuffled, then each one is
    handed to whichever split is currently furthest below its target share of
    segments. It is deterministic given SEED and needs no external dependency.
    '''
    rng = np.random.RandomState(seed)
    subjects = list(np.unique(subject_ids))
    rng.shuffle(subjects)
    counts = {s: int((subject_ids == s).sum()) for s in subjects}
    total = sum(counts.values())
    targets = {"train": train_frac, "val": val_frac, "test": test_frac}
    running = {"train": 0, "val": 0, "test": 0}
    assigned = {"train": [], "val": [], "test": []}
    for s in subjects:
        deficits = {k: targets[k] * total - running[k] for k in targets}
        best = max(deficits, key=lambda k: deficits[k])
        assigned[best].append(s)
        running[best] += counts[s]
    return assigned["train"], assigned["val"], assigned["test"]


In [ ]:
train_subjects, val_subjects, test_subjects = group_aware_pooled_split(subject_ids)

# Prove there is no leakage: each subject is in exactly one split.
assert set(train_subjects).isdisjoint(val_subjects), "LEAKAGE: a subject is in both train and val!"
assert set(train_subjects).isdisjoint(test_subjects), "LEAKAGE: a subject is in both train and test!"
assert set(val_subjects).isdisjoint(test_subjects), "LEAKAGE: a subject is in both val and test!"

train_mask = np.isin(subject_ids, train_subjects)
val_mask   = np.isin(subject_ids, val_subjects)
test_mask  = np.isin(subject_ids, test_subjects)

Xtr_sc_raw, Xtr_ie_raw, ytr = X_eeg[train_mask], X_ieeg[train_mask], y[train_mask]
Xva_sc_raw, Xva_ie_raw, yva = X_eeg[val_mask],   X_ieeg[val_mask],   y[val_mask]
Xte_sc_raw, Xte_ie_raw, yte = X_eeg[test_mask],  X_ieeg[test_mask],  y[test_mask]
str_te = subject_ids[test_mask]

print(f"Segments -> train={train_mask.sum()} ({100*train_mask.mean():.1f}%)  "
      f"val={val_mask.sum()} ({100*val_mask.mean():.1f}%)  "
      f"test={test_mask.sum()} ({100*test_mask.mean():.1f}%)   [target: 70% / 10% / 20%]")
print(f"Train subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Val subjects   ({len(val_subjects)}): {sorted(val_subjects)}")
print(f"Test subjects  ({len(test_subjects)}): {sorted(test_subjects)}")
print("No-leakage check PASSED: every subject appears in exactly one split (see asserts above).")

def zscore_fit(x):
    mu = x.mean(axis=(0, 1), keepdims=True)
    sd = x.std(axis=(0, 1), keepdims=True) + 1e-8
    return mu, sd

def zscore_apply(x, mu, sd):
    return (x - mu) / sd

sc_mu, sc_sd = zscore_fit(Xtr_sc_raw)
ie_mu, ie_sd = zscore_fit(Xtr_ie_raw)
Xtr_sc, Xva_sc, Xte_sc = (zscore_apply(a, sc_mu, sc_sd) for a in (Xtr_sc_raw, Xva_sc_raw, Xte_sc_raw))
Xtr_ie, Xva_ie, Xte_ie = (zscore_apply(a, ie_mu, ie_sd) for a in (Xtr_ie_raw, Xva_ie_raw, Xte_ie_raw))


## 8. Train one CNN-BiLSTM on the pooled (leakage-free) training set

In [ ]:
tr_loader = DataLoader(SegSet(Xtr_sc, Xtr_ie, ytr), batch_size=BATCH_SIZE, shuffle=True)
va_loader = DataLoader(SegSet(Xva_sc, Xva_ie, yva), batch_size=64, shuffle=False)
te_loader = DataLoader(SegSet(Xte_sc, Xte_ie, yte), batch_size=64, shuffle=False)

model = CNNBiLSTM(in_ch=M, out_ch=Mb, cnn_channels=CNN_CHANNELS, kernel_size=KERNEL_SIZE,
                   dilations=DILATIONS, lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS,
                   dropout=DROPOUT)

model = fit_model(model, tr_loader, va_loader, device=DEVICE, epochs=EPOCHS, lr=LR,
                   w_mse=W_MSE, w_pcorr=W_PCORR, w_cos=W_COS, w_l1=W_L1, w_amp=W_AMP,
                   patience=PATIENCE, verbose=True)

pooled_metrics = score_mapping(model, te_loader, device=DEVICE)
print(f"Test (unseen subjects) -> Combined: MSE={pooled_metrics['Combined']['MSE']:.3f} "
      f"PCORR={pooled_metrics['Combined']['PCORR']:.3f} COSSIM={pooled_metrics['Combined']['COSSIM']:.3f}")


## 9. Results table — overall pooled test set, plus a per-subject breakdown

In [ ]:
per_subject_metrics = {}
for subj in test_subjects:
    mask = str_te == subj
    if mask.sum() == 0:
        continue
    loader = DataLoader(SegSet(Xte_sc[mask], Xte_ie[mask], yte[mask]), batch_size=64, shuffle=False)
    per_subject_metrics[subj] = score_mapping(model, loader, device=DEVICE)
per_subject_metrics["Overall (pooled test set)"] = pooled_metrics

results_table = build_results_table(per_subject_metrics, index_name="Subject")
display(results_table)


### Per-channel breakdown (pooled test set)

A single poor channel can pull the Combined average down a lot even when most
channels are already matching well. This shows PCORR/COSSIM/MSE broken out by iEEG
channel over the whole pooled (unseen-subject) test set, sorted worst-to-best.


In [ ]:
def channel_breakdown(model, loader, device=DEVICE):
    '''Per-CHANNEL average MSE/PCORR/COSSIM (as opposed to score_mapping's overall
    average across every segment AND channel). A single very poor channel can pull a
    Combined average down substantially even when most channels are already matching
    well -- e.g. three channels at ~0.85 and one at ~0.20 average to only ~0.66. This
    shows whether that's happening, and if so, which channel(s) are the actual
    problem, rather than treating the model as uniformly bad.'''
    model.eval()
    per_ch = {j: {"mse": [], "pcorr": [], "cos": []} for j in range(Mb)}
    with torch.no_grad():
        for sc, ie, _ in loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for j in range(ie_np.shape[2]):
                for i in range(ie_np.shape[0]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    per_ch[j]["mse"].append(np.mean((yv - yev) ** 2))
                    per_ch[j]["pcorr"].append(pearsonr(yv, yev)[0])
                    denom = np.linalg.norm(yv) * np.linalg.norm(yev)
                    per_ch[j]["cos"].append(np.dot(yv, yev) / (denom + 1e-8))
    rows = {}
    for j in range(Mb):
        name = fo_names[j] if j < len(fo_names) else f"ch{j}"
        rows[name] = {
            "MSE": float(np.mean(per_ch[j]["mse"])),
            "PCORR": float(np.mean(per_ch[j]["pcorr"])),
            "COSSIM": float(np.mean(per_ch[j]["cos"])),
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index.name = "iEEG channel"
    return df.round(3).sort_values("PCORR")

display(channel_breakdown(model, te_loader, device=DEVICE))


## 10. Save results and models

In [ ]:
results_table.to_csv(os.path.join(".", "CNN_BILSTM_nbd_pooledsplit_results.csv"))
torch.save(model.state_dict(), os.path.join(".", "CNN_BILSTM_nbd_pooledsplit_model.pt"))
print("Saved: CNN_BILSTM_nbd_pooledsplit_results.csv, CNN_BILSTM_nbd_pooledsplit_model.pt")


## 11. Sanity check: real vs. estimated iEEG (unseen test subjects)

In [ ]:
def plot_real_vs_est(model, loader, ie_mu, ie_sd, ch=0, n=3, title=""):
    model.eval()
    sc, ie, lab = next(iter(loader))
    with torch.no_grad():
        y_est = model(sc.to(DEVICE)).cpu().numpy()
    ie_np = ie.numpy() * ie_sd + ie_mu          # back to physical units
    y_est = y_est * ie_sd + ie_mu
    fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
    for i in range(n):
        axes[i].plot(ie_np[i, :, ch], label="Real iEEG", color="black")
        axes[i].plot(y_est[i, :, ch], label="Estimated iEEG", color="crimson", alpha=0.8)
        lab_txt = f"label={int(lab[i].item())}" if HAS_LABELS else f"segment #{i}"
        axes[i].set_title(lab_txt)
        axes[i].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
te_loader_shuf = DataLoader(SegSet(Xte_sc, Xte_ie, yte), batch_size=8, shuffle=True)
plot_real_vs_est(model, te_loader_shuf, ie_mu, ie_sd, ch=0,
                  title=f"Pooled split (unseen test subjects) - iEEG channel {fo_names[0]}")


## 12. Best-case vs. typical-case matching (pooled test set)


In [ ]:
sc_batch = torch.tensor(Xte_sc[:300], dtype=torch.float32)
ie_batch = torch.tensor(Xte_ie[:300], dtype=torch.float32)
lab_batch = torch.tensor(yte[:300], dtype=torch.float32)
plot_best_vs_typical(model, sc_batch, ie_batch, lab_batch, ie_mu, ie_sd, fo_names,
                      n_examples=4, title="Pooled test set (unseen subjects)")
